# Reranking with cross-encoders: the real-library code of the lessons

Booster LEAP, Advanced RAG Engineering. The cells below are the library versions of the examples in the lessons (the lessons first build the same mechanism in plain Python in your browser). Each cell was run once and its output is saved below it.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use it:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again.

**What needs what:** the first run downloads two small models, all-MiniLM-L6-v2 and cross-encoder/ms-marco-MiniLM-L-6-v2 (about 90 MB each; it needs internet but no GPU). No secret or key is used in this notebook. The install line pins the versions that were used to produce the saved outputs; Colab may resolve the dependencies of those packages differently.

In [ ]:
%pip -q install sentence-transformers==6.3.0 numpy==2.5.2

## Two-stage retrieval: retrieve, then rerank

CrossEncoder scores the dense top 5 of two questions: the same order and scores as the saved results.

In [ ]:
docs = {
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
}
dense_ranks = {
    "q01": ["kb-err4013", "kb-err4021", "kb-err5032", "kb-err5103", "kb-err5031"],
    "q12": ["exp-travel", "faq-seats", "kb-merge", "kb-err5031", "faq-trash"],
}
queries = {
    "q01": "ERR-5103",
    "q12": "can I fly in a fancier seat",
}

from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
for q, candidates in dense_ranks.items():
    pairs = [(queries[q], docs[p]) for p in candidates]          # one pair per candidate
    scores = model.predict(pairs, show_progress_bar=False)
    order = sorted(zip(candidates, scores), key=lambda x: -x[1])
    print(queries[q])
    for i, (p, s) in enumerate(order, 1):
        print(f"  {i}. {p:13s} {s:7.2f}")

ERR-5103
  1. kb-err5103       7.70
  2. kb-err4013       0.92
  3. kb-err4021      -0.99
  4. kb-err5032      -1.50
  5. kb-err5031      -1.63
can I fly in a fancier seat
  1. faq-seats      -11.18
  2. exp-travel     -11.33
  3. kb-merge       -11.36
  4. faq-trash      -11.38
  5. kb-err5031     -11.41


## Cross-encoder against bi-encoder

Both models on one pair: two vectors from the bi-encoder, one combined input and one score from the cross-encoder.

In [ ]:
from sentence_transformers import CrossEncoder, SentenceTransformer

bi = SentenceTransformer("all-MiniLM-L6-v2")
cross = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

query, passage = "who bit the man", "The dog bit the man."
vectors = bi.encode([query, passage])                    # two separate inputs
print("bi-encoder output:", vectors.shape[0], "vectors of", vectors.shape[1], "numbers each")
print("lengths of the vectors:", [round(float((v * v).sum()) ** 0.5, 3) for v in vectors])

ids = cross.tokenizer(query, passage)["input_ids"]       # one combined input
print("cross-encoder input:", cross.tokenizer.decode(ids))
print("cross-encoder output:", cross.predict([(query, passage)], show_progress_bar=False).shape[0], "score")

bi-encoder output: 2 vectors of 384 numbers each
lengths of the vectors: [1.0, 1.0]
cross-encoder input: [CLS] who bit the man [SEP] the dog bit the man. [SEP]
cross-encoder output: 1 score


Four questions, each with two passages that differ in word order or in a negation: bi-encoder cosine and raw cross-encoder score.

In [ ]:
from sentence_transformers import CrossEncoder, SentenceTransformer

bi = SentenceTransformer("all-MiniLM-L6-v2")
cross = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

cases = {
    "who bit the man": ["The dog bit the man.", "The man bit the dog."],
    "who bit the dog": ["The dog bit the man.", "The man bit the dog."],
    "flight from Delhi to Mumbai": ["This flight goes from Delhi to Mumbai.", "This flight goes from Mumbai to Delhi."],
    "can I carry over unused leave": ["You can carry over unused leave.", "You cannot carry over unused leave."],
}
for query, passages in cases.items():
    q, *p = bi.encode([query] + passages)                # vectors of length 1, so a dot product is the cosine
    scores = cross.predict([(query, x) for x in passages], show_progress_bar=False)
    print(query)
    for text, vector, score in zip(passages, p, scores):
        print(f"  bi {float(q @ vector):.3f}  cross {score:6.2f}  {text}")

who bit the man
  bi 0.693  cross   7.84  The dog bit the man.
  bi 0.702  cross   5.70  The man bit the dog.
who bit the dog
  bi 0.837  cross   6.86  The dog bit the man.
  bi 0.856  cross   8.44  The man bit the dog.
flight from Delhi to Mumbai
  bi 0.921  cross   8.10  This flight goes from Delhi to Mumbai.
  bi 0.917  cross   7.91  This flight goes from Mumbai to Delhi.
can I carry over unused leave
  bi 0.926  cross   9.93  You can carry over unused leave.
  bi 0.915  cross  10.51  You cannot carry over unused leave.


## Reranker models and hosting

Look inside the model you are about to deploy instead of guessing. Check the version you use: attribute names differ between releases of sentence-transformers.

In [ ]:
from sentence_transformers import CrossEncoder

NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
REVISION = "233902d25c440f23af6f7d6e94d2946bac0bee0a"      # pin the exact version you tested
model = CrossEncoder(NAME, revision=REVISION)
print("maximum input length in tokens:", model.max_seq_length)
print("numbers returned per pair:", model.num_labels)
print("function applied to the output by predict():", type(model.activation_fn).__name__)
print("parameters:", sum(p.numel() for p in model.model.parameters()))

maximum input length in tokens: 512
numbers returned per pair: 1
function applied to the output by predict(): Identity
parameters: 22713601


The same answer sentence at the start or at the end of a longer and longer passage of unrelated text.

In [ ]:
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", revision="233902d25c440f23af6f7d6e94d2946bac0bee0a")
query = "Sev 2 response time"
answer = "Sev 2: respond within 30 minutes."
filler = "The cafeteria menu changes every week and the lifts are checked on Monday. "

for times in (0, 5, 30, 40):
    first = answer + " " + filler * times          # the answer at the start of the passage
    last = filler * times + answer                 # the answer at the end of the passage
    tokens = len(model.tokenizer(query, last)["input_ids"])
    a, b = model.predict([(query, first), (query, last)], show_progress_bar=False)
    print(f"filler x{times:2}: {tokens:3} tokens | answer first {a:6.2f} | answer last {b:6.2f}")
alone = model.predict([(query, filler * 40)], show_progress_bar=False)[0]
print(f"filler x40 with no answer at all: {alone:6.2f}")

filler x 0:  17 tokens | answer first   9.05 | answer last   9.05
filler x 5:  87 tokens | answer first   7.52 | answer last   4.64
filler x30: 437 tokens | answer first   4.97 | answer last  -6.85
filler x40: 577 tokens | answer first   4.61 | answer last -10.22
filler x40 with no answer at all: -10.22


One pair, then 32 pairs singly and as one batch. Timings differ from run to run and from machine to machine.

In [ ]:
import statistics
import time
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", revision="233902d25c440f23af6f7d6e94d2946bac0bee0a")
pair = ("how much time off do I get each year",
        "Full-time employees receive 24 days of paid annual leave per calendar year.")
model.predict([pair], show_progress_bar=False)      # warm-up: the first call is slower

def milliseconds(count, batch_size, repeats=7):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        model.predict([pair] * count, batch_size=batch_size, show_progress_bar=False)
        times.append(time.perf_counter() - start)
    return 1000 * statistics.median(times)

for count, batch_size in [(1, 1), (32, 1), (32, 32)]:
    ms = milliseconds(count, batch_size)
    print(f"pairs {count:2}, batch size {batch_size:2}: {ms:.1f} ms, {ms / count:.2f} ms per pair")

pairs  1, batch size  1: 13.5 ms, 13.51 ms per pair
pairs 32, batch size  1: 399.0 ms, 12.47 ms per pair
pairs 32, batch size 32: 137.1 ms, 4.28 ms per pair


## Listwise and model-based rerankers

Pointwise scoring and an all-pairs tournament built from the same scores.

In [ ]:
import itertools
from sentence_transformers import CrossEncoder

docs = {
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
}

query = "ERR-5103"
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", revision="233902d25c440f23af6f7d6e94d2946bac0bee0a")
ids = list(docs)
scores = dict(zip(ids, model.predict([(query, docs[i]) for i in ids], show_progress_bar=False)))
pointwise = sorted(ids, key=lambda i: -scores[i])
print("pointwise, calls:", len(ids))
for i in pointwise:
    print(f"  {i:11s} {scores[i]:6.2f}")

wins, comparisons = {i: 0 for i in ids}, 0
for a, b in itertools.combinations(ids, 2):          # every pair once
    wins[a if scores[a] >= scores[b] else b] += 1
    comparisons += 1
tournament = sorted(ids, key=lambda i: -wins[i])
print("all pairs, comparisons:", comparisons, "| wins:", [wins[i] for i in tournament])
print("same order as pointwise:", tournament == pointwise)

pointwise, calls: 5
  kb-err5103    7.70
  kb-err4013    0.92
  kb-err4021   -0.99
  kb-err5032   -1.50
  kb-err5031   -1.63
all pairs, comparisons: 10 | wins: [4, 3, 2, 1, 0]
same order as pointwise: True


## Choosing how many candidates to rerank

Time the real cross-encoder for four values of n, 20 calls each.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}

import time
from sentence_transformers import CrossEncoder

model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
query = "how much time off do I get each year"
texts = list(docs.values())

def percentile(values, p):
    ordered = sorted(values)
    return ordered[max(0, -(-p * len(ordered) // 100) - 1)]        # nearest-rank percentile

model.predict([(query, texts[0])], show_progress_bar=False)         # warm up
for n in (1, 5, 10, 28):
    times = []
    for _ in range(20):
        start = time.perf_counter()
        model.predict([(query, t) for t in texts[:n]], batch_size=32, show_progress_bar=False)
        times.append((time.perf_counter() - start) * 1000)
    print(f"n={n:2d}  median {percentile(times, 50):6.1f} ms  p95 {percentile(times, 95):6.1f} ms")

n= 1  median   14.3 ms  p95   15.9 ms
n= 5  median   36.6 ms  p95   45.0 ms
n=10  median   80.8 ms  p95   86.9 ms
n=28  median  210.9 ms  p95  230.3 ms


## Notebook: rerank from end to end

predict and rank on three queries, checked against the saved results.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
queries = {
    "q01": "ERR-5103",
    "q12": "can I fly in a fancier seat",
    "h04": "can the other parent take time off after the birth",
}
# saved results to compare with: candidates and cross-encoder scores
expected = {
    "q01": {"kb-err4013": 0.9229, "kb-err4021": -0.9906, "kb-err5032": -1.4984, "kb-err5103": 7.7031, "kb-err5031": -1.63},
    "q12": {"exp-travel": -11.327, "faq-seats": -11.1802, "kb-merge": -11.3614, "kb-err5031": -11.4114, "faq-trash": -11.3796},
    "h04": {"leave-parental": -8.1605, "leave-apply": -11.3163, "leave-carry": -11.3341, "leave-annual": -11.3333, "kb-merge": -11.3862},
}

from sentence_transformers import CrossEncoder, SentenceTransformer

bi = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
cross = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
ids = list(docs)
vectors = bi.encode([docs[i] for i in ids], normalize_embeddings=True, show_progress_bar=False)   # computed once

for qid, query in queries.items():
    cosines = vectors @ bi.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    candidates = [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-float(cosines[j]), ids[j]))[:5]]
    assert candidates == list(expected[qid]), (qid, candidates)

    scores = cross.predict([(query, docs[p]) for p in candidates], show_progress_bar=False)
    assert all(abs(float(s) - expected[qid][p]) < 0.02 for p, s in zip(candidates, scores)), qid

    ranked = cross.rank(query, [docs[p] for p in candidates], top_k=3)       # the same thing in one call
    top3 = [candidates[hit["corpus_id"]] for hit in ranked]
    assert top3 == sorted(candidates, key=lambda p: -expected[qid][p])[:3], (qid, top3)
    print(qid, query)
    for hit in ranked:
        print(f"    {float(hit['score']):7.2f}  {candidates[hit['corpus_id']]}")
print("candidates and scores equal the saved ones")

q01 ERR-5103
       7.70  kb-err5103
       0.92  kb-err4013
      -0.99  kb-err4021
q12 can I fly in a fancier seat
     -11.18  faq-seats
     -11.33  exp-travel
     -11.36  kb-merge
h04 can the other parent take time off after the birth
      -8.16  leave-parental
     -11.32  leave-apply
     -11.33  leave-annual
candidates and scores equal the saved ones


The whole report with the real models, and the time per search.

In [ ]:
docs = {
    "leave-annual": "Full-time employees receive 24 days of paid annual leave per calendar year. Public holidays are not counted.",
    "leave-sick": "Employees may take up to 10 paid sick days per calendar year. From the third consecutive sick day a medical certificate is required.",
    "leave-parental": "The primary caregiver receives 26 weeks of paid parental leave and the second caregiver receives 6 weeks.",
    "leave-carry": "Up to 5 unused annual leave days may be carried over and must be used by 31 March.",
    "leave-apply": "Request leave in the HR portal at least 5 working days before the first day of leave.",
    "exp-meals": "Daily meal limits in rupees: Tier 1 city 400 breakfast, 700 lunch, 1200 dinner. Tier 2 city 300, 500, 900. Tier 3 city 250, 400, 700.",
    "exp-approval": "Claims above 5000 rupees need a receipt and manager approval. Claims above 25000 rupees also need finance approval.",
    "exp-travel": "Book economy class for flights shorter than 6 hours. Business class needs written approval from the department head.",
    "sec-passwords": "Passwords must have at least 14 characters and must not be reused. Multi-factor authentication is required for email and the cloud console.",
    "sec-devices": "Report a lost or stolen laptop to the security team within 2 hours. Company laptops lock after 5 minutes without activity.",
    "sec-data": "Never copy customer data into chat tools, personal notes or public AI services.",
    "faq-seats": "The Starter plan includes up to 5 seats. Teams that need more seats move to the Team plan.",
    "faq-sso": "Single sign-on with SAML is available on the Business plan and above.",
    "faq-trash": "Deleted tickets stay in the trash for 30 days and are then removed for good.",
    "faq-region": "Customer data is stored in the Mumbai region by default. Business plan customers may choose Frankfurt.",
    "faq-uptime": "The Business plan commits to 99.9 percent monthly uptime, with a credit of 10 percent of the month's fee if it falls below.",
    "run-sev": "Sev 1: the service is down for most customers. Respond within 5 minutes. Sev 2: respond within 30 minutes.",
    "run-rollback": "If the problem started right after a release, roll back first. Command: orbit deploy rollback --service desk-api --to previous.",
    "kb-err4012": "Error ERR-4012 at sign-in: the SAML certificate has expired. An admin uploads the new certificate in Settings, then Security.",
    "kb-err4013": "Error ERR-4013 at sign-in: the password was typed wrongly too many times. The account unlocks by itself after 15 minutes.",
    "kb-err4021": "Error ERR-4021 at sign-in: the authenticator code is out of date. Check that the phone clock is set to automatic time.",
    "kb-err5031": "Error ERR-5031 on export: the export file is larger than 2 GB. Filter the tickets by date and export in several parts.",
    "kb-err5032": "Error ERR-5032 on export: the export link has expired after 24 hours. Start a new export in Settings, then Data.",
    "kb-err5103": "Error ERR-5103 on import: the CSV file has a column that is not in the template. Remove the extra column and upload again.",
    "kb-login-loop": "If the sign-in page keeps reloading, clear the browser cookies for orbitdesk.example and try again.",
    "kb-slow": "Pages load slowly when the inbox holds more than 10,000 open tickets. Archive old tickets to speed it up.",
    "kb-mail": "Customer emails missing from the inbox are usually in the spam filter. Add the sender to the allow list in Settings, then Mail.",
    "kb-merge": "Two tickets from the same customer can be merged. The older ticket keeps its number.",
}
queries = {
    "q01": "ERR-5103",
    "q02": "ERR-5031",
    "q03": "ERR-4012",
    "q04": "Sev 2 response time",
    "q05": "Frankfurt",
    "q06": "10,000 tickets",
    "q07": "Tier 2 dinner limit",
    "q08": "31 March",
    "q09": "how much time off do I get each year",
    "q10": "can I paste client information into ChatGPT",
    "q11": "I am expecting a baby, how long can I stay home",
    "q12": "can I fly in a fancier seat",
    "q13": "messages from customers do not show up",
    "q14": "the app is very slow",
    "q15": "the website keeps showing the login again and again",
    "q16": "what if my work computer is taken",
    "h01": "ERR-4013",
    "h02": "99.9 uptime",
    "h03": "how long can I still recover deleted items",
    "h04": "can the other parent take time off after the birth",
    "h05": "ERR-5032 link",
    "h06": "who approves a first class flight",
    "h07": "Starter seats",
    "h08": "are my emails going to junk",
}
gold = {"q01": "kb-err5103", "q02": "kb-err5031", "q03": "kb-err4012", "q04": "run-sev", "q05": "faq-region", "q06": "kb-slow", "q07": "exp-meals", "q08": "leave-carry", "q09": "leave-annual", "q10": "sec-data", "q11": "leave-parental", "q12": "exp-travel", "q13": "kb-mail", "q14": "kb-slow", "q15": "kb-login-loop", "q16": "sec-devices", "h01": "kb-err4013", "h02": "faq-uptime", "h03": "faq-trash", "h04": "leave-parental", "h05": "kb-err5032", "h06": "exp-travel", "h07": "faq-seats", "h08": "kb-mail"}
EXPECTED = {"tuning": [0.9375, 1.0, 0.9688], "held-out": [1.0, 1.0, 1.0]}     # [recall@1, recall@3, MRR] after reranking

import time
from sentence_transformers import CrossEncoder, SentenceTransformer

bi = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
cross = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
ids = list(docs)
vectors = bi.encode([docs[i] for i in ids], normalize_embeddings=True, show_progress_bar=False)

def search(query, n=5):
    cosines = vectors @ bi.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    candidates = [ids[j] for j in sorted(range(len(ids)), key=lambda j: (-float(cosines[j]), ids[j]))[:n]]
    return [candidates[hit["corpus_id"]] for hit in cross.rank(query, [docs[p] for p in candidates])]

search("warm up")
times, results = [], {}
for qid, query in queries.items():
    start = time.perf_counter()
    results[qid] = search(query)
    times.append((time.perf_counter() - start) * 1000)

for label, prefix in (("tuning", "q"), ("held-out", "h")):
    qids = [q for q in queries if q.startswith(prefix)]
    ranks = [results[q].index(gold[q]) + 1 if gold[q] in results[q] else None for q in qids]
    report = [sum(r == 1 for r in ranks) / len(qids), sum(r is not None and r <= 3 for r in ranks) / len(qids),
              sum(1 / r for r in ranks if r) / len(qids)]
    report = [round(x, 4) for x in report]
    assert report == EXPECTED[label], (label, report)
    print(f"{label}: recall@1 {report[0]}  recall@3 {report[1]}  MRR {report[2]}")
times.sort()
print(f"{len(times)} searches: median {times[len(times) // 2]:.1f} ms, slowest {times[-1]:.1f} ms")

tuning: recall@1 0.9375  recall@3 1.0  MRR 0.9688
held-out: recall@1 1.0  recall@3 1.0  MRR 1.0
24 searches: median 46.9 ms, slowest 50.9 ms
